In [ ]:
import pymcel as pc 
import matplotlib.pyplot as plt
import numpy as np 
from mpl_toolkits.mplot3d import Axes3D

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='erfa')

### 1. Configuración e inicialización 

In [ ]:
#fecha un poco antes del encuentro para tener una referencia de la órbita de Apophis 
fecha_ref = '2029-01-01 00:00:00'

#Extraer de pymcel (están en m^3/s^2) y convertir a km^3/s^2 para consistencia con las unidades de posición y velocidad que usaremos (km y km/s).
mu_sol = pc.constantes.mu_sun /1e9
mu_tierra = pc.constantes.mu_earth /1e9
mu_luna = pc.constantes.mu_moon /1e9
mu_jupiter = pc.constantes.mu_jupiter /1e9

def estado(id_obj):
    _, _, X = pc.consulta_horizons( id = id_obj, location = '@sun', epochs = fecha_ref) #obtenemos el estado del objeto en la fecha de referencia
    dist = np.linalg.norm(X[:3]) #calculamos la distancia al Sol

    #Si la distancia al Sol es > 1e10, está en metros, así que convertimos a km
    if dist > 1e10:
        X /= 1e3 #convertimos posición a km 
    #Si la distancia al Sol es < 100, está en UA, así que convertimos a km
    elif dist < 100:
        X[:3] *= 149597870.7 #convertimos posición a km
        X[3:] *= 149597870.7 / 86400 #convertimos velocidad a km/s (1 UA/día a km/s)
    return X[:3], X[3:] #entrega posición [km] y velocidad [km/s]


#obtenemos el estado de los objetos
r_sol = np.zeros(3) #el Sol está en el origen

r_tierra, v_tierra = estado('399') #estado de la Tierra
r_luna, v_luna = estado('301') #estado de la Luna
r_jupiter, v_jupiter = estado('599') #estado de Júpiter
r_apophis, v_apophis = estado('99942') #estado de Apophis

### 2. Aplicar la integración numérica de $N$-cuerpos

Para resolver las ecuaciones de movimiento del sistema se implementará el algoritmo de integración numérica **Leapfrog**.

* Funcionamiento matemático

La estructura utilizada sigue el esquema de tres sub-pasos por cada intervalo de tiepo $\Delta t$:

1. **Medio paso de posición:** Se actualiza la posición del cuerpo hasta la mitad del intervalo temporal usando la velocidad actual: 

$$r( t +\frac{1}{2} \Delta t) = r(t) + v(t) \frac{1}{2} \Delta t$$

2. **Paso de velocidad**: Se calcula la aceleración gravitatoria $a$ en la nueva posición media y se actualiza la velocidad completa para todo el intervalo: 

$$v(t+ \Delta t) = v(t) + a(t + \frac{1}{2} \Delta t) \Delta t$$

3. **Medio paso de posición**: Se completa la actualización de la posición usando la nueva velocidad: 

$$r(t + \Delta t) = r( t +\frac{1}{2} \Delta t) + v(t + \Delta t) \frac{1}{2} \Delta t$$

In [ ]:
#El tiempo son 5 meses desde enero hasta mayo
delta_t = 30 #segundos
dias_simulacion = 150 #días
N_steps = int((dias_simulacion*86400)/delta_t) #Número de pasos para 150 días

pos_tierra = np.zeros((N_steps, 3))
pos_apophis = np.zeros((N_steps, 3))
distancias = np.zeros(N_steps)

#Método de integración Leapfrog para la simulación de la órbita 
for i in range(N_steps):
    #Distancia relatica Apophis - Tierra
    pos_tierra[i] = r_tierra
    pos_apophis[i] = r_apophis
    distancias[i] = np.linalg.norm(r_apophis - r_tierra)

    # DRIFT (Medio paso de posición)
    r_ap_half = r_apophis + v_apophis * (delta_t*0.5)
    r_tr_half = r_tierra + v_tierra * (delta_t*0.5)
    r_ln_half = r_luna + v_luna * (delta_t*0.5)
    r_ju_half = r_jupiter + v_jupiter * (delta_t*0.5)

    #Función aceleración 
    def aceleracion(p_obj, p_atr, mu_atr):
        r_vec = p_obj - p_atr
        r_mag = np.linalg.norm(r_vec)
        return -mu_atr * r_vec / r_mag**3
    
    #KICK (Paso completo de velocidad)
    a_ap = aceleracion(r_ap_half, r_sol, mu_sol) + aceleracion(r_ap_half, r_tr_half, mu_tierra) \
        + aceleracion(r_ap_half, r_ln_half, mu_luna) + aceleracion(r_ap_half, r_ju_half, mu_jupiter)
    
    a_tr = aceleracion(r_tr_half, r_sol, mu_sol) + aceleracion(r_tr_half, r_ju_half, mu_jupiter)
    a_ln = aceleracion(r_ln_half, r_sol, mu_sol) + aceleracion(r_ln_half, r_tr_half, mu_tierra)
    a_ju = aceleracion(r_ju_half, r_sol, mu_sol)

    #Actualizamos velocidades
    v_apophis += a_ap * delta_t
    v_tierra += a_tr * delta_t
    v_luna += a_ln * delta_t
    v_jupiter += a_ju * delta_t

    #DRIFT (Medio paso de posición)
    r_apophis = r_ap_half + v_apophis * (delta_t*0.5)
    r_tierra = r_tr_half + v_tierra * (delta_t*0.5)
    r_luna = r_ln_half + v_luna * (delta_t*0.5)
    r_jupiter = r_ju_half + v_jupiter * (delta_t*0.5)

### 3. Cálculo de resultados

In [ ]:
idx_min = np.argmin(distancias)
print(f"Distancia mínima: {distancias[idx_min]:,.2f} km")
print(f"Altitud: {distancias[idx_min] - 6371:,.2f} km")
print(f"Día del perigeo: {idx_min * delta_t / 86400:.2f} días desde el 1 de Enero")

### 4. Gráfica 


In [ ]:
fig = plt.figure(figsize=(15, 6))

#Gráfica 3D de las órbitas
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot(pos_tierra[:, 0], pos_tierra[:, 1], pos_tierra[:, 2], label='Tierra', color = 'blue')
ax1.plot(pos_apophis[:, 0], pos_apophis[:, 1], pos_apophis[:, 2], label='Apophis', color = 'red')
ax1.scatter(0, 0, 0, color='orange', label='Sol', s=100)

lim = 1.8e8 
ax1.set_xlim(-lim, lim); ax1.set_ylim(-lim, lim); ax1.set_zlim(-lim, lim)
ax1.set_title('Órbitas de la Tierra y Apophis')
ax1.set_xlabel('X (km)')
ax1.set_ylabel('Y (km)')
ax1.set_zlabel('Z (km)')
ax1.legend()

#Gráfia de la distancia Tierra-Apophis
ax2 = fig.add_subplot(122)
tiempo_dias = np.linspace(0, dias_simulacion, N_steps)
ax2.plot(tiempo_dias, distancias / 1e6, color='darkgreen', lw=2)
ax2.plot(np.linspace(0, dias_simulacion, N_steps), distancias / 1e6, color='green')
ax2.scatter(tiempo_dias[idx_min], idx_min / 1e6, color='red', zorder=5)
ax2.set_title("Distancia Tierra - Apophis")
ax2.set_xlabel('Tiempo (días)')
ax2.set_ylabel('Distancia (millones de km)')
ax2.grid(True)

plt.show()

La simulación de largo alcance ($150$ días) valida con éxtio la trayectoria heliocéntrica y predice corretamente la ventana del encuentro cercano al 13 de abril de 2029 (103 días desde el 1 de enero). La discrepancia entre la altitud simulada ($173,731 km$) y la oficial ($31,600 km$) puede deberse a la acumulación de error numérico inherente a la integración de largo plazo y a la exclusión de perturbaciones planetarias menores, confirmando la necesidad de modelos más precisos.